# GPT-2 Minmax Experiment — Full Comparison (Colab)

Single repo runs **all four** 500-step training conditions:

| Framework | Condition | Log file |
|-----------|-----------|----------|
| **TRL** | Baseline (KL=0.2) | `ppo_minmax_experiment/results/baseline_training_log.csv` |
| **TRL** | Minmax (KL=0.01) | `ppo_minmax_experiment/results/minmax_training_log.csv` |
| **safe-rlhf** | Baseline (KL=0.2) | `output/gpt2-baseline-500/baseline_training_log.csv` |
| **safe-rlhf** | Minmax (KL=0.01) | `output/gpt2-minmax-500/minmax_training_log.csv` |

**Repo:** https://github.com/wendymaboa/urban-spoon

**One command for everything:** `bash scripts/run-all-gpt2-500.sh`

**Recommended workflow:** (1) run safe-rlhf baseline + minmax, (2) `diagnose_unsafe_rate.py` to check penalty engagement, (3) re-run TRL minmax only if unsafe rate is sufficient.

In [ ]:
#@title 0. GPU check
!nvidia-smi

In [ ]:
#@title 1. Clone repo and install dependencies
from pathlib import Path

REPO_URL = "https://github.com/wendymaboa/urban-spoon.git"
BRANCH = "main"
ROOT = Path("/content/urban-spoon")

if ROOT.exists():
    !cd {ROOT} && git pull
else:
    !git clone -b {BRANCH} {REPO_URL} {ROOT}

%cd {ROOT}
# Colab pre-installs transformers 5.x + trl 0.27+ which break our scripts.
!pip install -q --upgrade pip
!pip install -q "transformers>=4.37.2,<=4.46.3" "trl==0.11.4" "tokenizers>=0.13.3,<0.22.0"
!pip install -q -r requirements.txt
!pip install -q -e . --no-deps
!python -c "import transformers, trl; print('transformers', transformers.__version__); print('trl', trl.__version__)"

In [ ]:
#@title 2. Mount Google Drive (optional — saves logs + models)
from google.colab import drive
drive.mount('/content/drive')

USE_DRIVE = True
DRIVE_DIR = "/content/drive/MyDrive/RLHC_Experiments/gpt2_500steps"
!mkdir -p "{DRIVE_DIR}"

In [ ]:
#@title 3. Config — which experiments to run
MAX_STEPS = 500

# Toggle individual runs (all True = full 4-way comparison)
RUN_TRL = False         # ppo_minmax_experiment (TRL) — re-enable after diagnose_unsafe_rate
RUN_SAFE_RLHF = True    # safe-rlhf (DeepSpeed) — prioritize for valid comparison

SKIP_TRL_BASELINE = True   # TRL baseline already completed
SKIP_TRL_MINMAX = True     # re-run Minmax only after unsafe-rate diagnosis
SKIP_SAFE_RLHF_BASELINE = False
SKIP_SAFE_RLHF_MINMAX = False

# Set True to run everything via one shell script:
RUN_ALL_VIA_SCRIPT = False

In [ ]:
#@title 3b. Diagnose unsafe rate (optional — run before TRL Minmax)
RUN_DIAGNOSE = False
DIAG_BATCHES = 25
SAFETY_THRESHOLD = -0.3

if RUN_DIAGNOSE:
    !python ppo_minmax_experiment/diagnose_unsafe_rate.py \
        --batches {DIAG_BATCHES} \
        --safety-threshold {SAFETY_THRESHOLD} \
        --use-ppo-trainer

In [ ]:
#@title 4. Run experiments
import os

os.environ["MAX_TRAINING_STEPS"] = str(MAX_STEPS)

if RUN_ALL_VIA_SCRIPT:
    os.environ["RUN_TRL"] = "1" if RUN_TRL else "0"
    os.environ["RUN_SAFE_RLHF"] = "1" if RUN_SAFE_RLHF else "0"
    os.environ["SKIP_TRL_BASELINE"] = "1" if SKIP_TRL_BASELINE else "0"
    os.environ["SKIP_TRL_MINMAX"] = "1" if SKIP_TRL_MINMAX else "0"
    os.environ["SKIP_SAFE_RLHF_BASELINE"] = "1" if SKIP_SAFE_RLHF_BASELINE else "0"
    os.environ["SKIP_SAFE_RLHF_MINMAX"] = "1" if SKIP_SAFE_RLHF_MINMAX else "0"
    !bash scripts/run-all-gpt2-500.sh
else:
    if RUN_TRL:
        skip_b = "--skip-baseline" if SKIP_TRL_BASELINE else ""
        skip_m = "--skip-minmax" if SKIP_TRL_MINMAX else ""
        !python ppo_minmax_experiment/run_experiment.py --only train --steps {MAX_STEPS} {skip_b} {skip_m}

    if RUN_SAFE_RLHF:
        os.environ.pop("SKIP_BASELINE", None)
        os.environ.pop("SKIP_MINMAX", None)
        if SKIP_SAFE_RLHF_BASELINE:
            os.environ["SKIP_BASELINE"] = "1"
        if SKIP_SAFE_RLHF_MINMAX:
            os.environ["SKIP_MINMAX"] = "1"
        !bash scripts/run-gpt2-500-comparison.sh

In [ ]:
#@title 5. Copy outputs to Drive
if USE_DRIVE:
    !cp -r ppo_minmax_experiment/results "{DRIVE_DIR}/trl_results" 2>/dev/null || true
    !cp -r output/gpt2-baseline-500 "{DRIVE_DIR}/" 2>/dev/null || true
    !cp -r output/gpt2-minmax-500 "{DRIVE_DIR}/" 2>/dev/null || true
    print(f"Saved to {DRIVE_DIR}")

In [ ]:
#@title 6. Compare all four training logs
import pandas as pd
from pathlib import Path

logs = [
    ("TRL baseline", "ppo_minmax_experiment/results/baseline_training_log.csv"),
    ("TRL Minmax", "ppo_minmax_experiment/results/minmax_training_log.csv"),
    ("safe-rlhf baseline", "output/gpt2-baseline-500/baseline_training_log.csv"),
    ("safe-rlhf Minmax", "output/gpt2-minmax-500/minmax_training_log.csv"),
]

rows = []
for name, path in logs:
    p = Path(path)
    if not p.exists():
        print(f"Missing: {path}")
        continue
    df = pd.read_csv(p)
    last = df.iloc[-1]
    rows.append({
        "run": name,
        "steps": len(df),
        "final_mean_reward": last.get("mean_reward"),
        "final_kl": last.get("kl_divergence"),
        "final_r_unsafe": last.get("r_unsafe", "N/A"),
    })

if rows:
    display(pd.DataFrame(rows))

In [ ]:
#@title 7. Optional — evaluate + plot (TRL pipeline)
RUN_EVAL = False
RUN_PLOTS = False

if RUN_EVAL:
    !python ppo_minmax_experiment/run_experiment.py --only eval
if RUN_PLOTS:
    !python ppo_minmax_experiment/run_experiment.py --only plot